In [ ]:
# Install (مرة واحدة فقط)
# pip install mediapipe opencv-python numpy

import cv2
import numpy as np
import time
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# =========================
# 1. فتح الكاميرا
# =========================
cap = cv2.VideoCapture(0)

# =========================
# 2. تحميل الموديل
# =========================
model_path = "hand_landmarker.task"  # عدلي المسار لو عندك مكان مختلف

base_options = python.BaseOptions(model_asset_path=model_path)

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    num_hands=1
)

landmarker = vision.HandLandmarker.create_from_options(options)

# =========================
# 3. Gesture Function
# =========================
def get_gesture(landmarks):
    tips = [8, 12, 16, 20]
    fingers = []

    # thumb
    fingers.append(1 if landmarks[4].x < landmarks[3].x else 0)

    # other fingers
    for tip in tips:
        fingers.append(1 if landmarks[tip].y < landmarks[tip - 2].y else 0)

    total = fingers.count(1)

    if total == 0:
        return "STOP ✊"
    elif total == 5:
        return "START ✋"
    elif fingers == [0,1,0,0,0]:
        return "NEXT ☝️"
    elif fingers == [0,1,1,0,0]:
        return "PAUSE ✌️"
    else:
        return "UNKNOWN"

# =========================
# 4. Main Loop
# =========================
frame_timestamp_ms = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    results = landmarker.detect_for_video(mp_image, frame_timestamp_ms)
    frame_timestamp_ms += 33

    gesture = "No Hand"

    if results.hand_landmarks:
        for hand in results.hand_landmarks:
            gesture = get_gesture(hand)

            for lm in hand:
                cx, cy = int(lm.x * w), int(lm.y * h)
                cv2.circle(frame, (cx, cy), 4, (0, 255, 0), -1)

    # show gesture text
    cv2.putText(frame, gesture, (50, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)

    cv2.imshow("Hand Gesture Recognition", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

    time.sleep(0.03)

# =========================
# 5. Release
# =========================
cap.release()
cv2.destroyAllWindows()